# Swin Transformer — Multi-Label Sınıflandırma (Fold 0)

**Kaynak notebook:** `cse521-swin-transformer-supervised.ipynb`  
**Veri seti:** TR_ABDOMEN_RAD_EMERGENCY — 6 acil karın patolojisi

### Orijinal → Uyarlama farkları
| Bileşen | Orijinal (Kaggle CT-Kidney) | Bu notebook |
|---|---|---|
| Veri formatı | PNG/JPEG → `ImageFolder` | DICOM → 3-kanallı HU NPZ |
| Sınıf | 4 sınıf, single-label | 6 sınıf, **multi-label** |
| Kayıp | `CrossEntropyLoss` | `FocalBCE` (class-balanced) |
| Tahmin | `softmax + argmax` | `sigmoid > 0.5` |
| Metrik | Accuracy | **mAP + macro-F1** |
| Model | `swin_tiny` | `swin_base` (ImageNet-22k pretrained) |
| Giriş boyutu | 224×224 | **384×384** (window12) |

**Ön koşullar:**
- `Faz1_Veri_Hazirlik_1fold.ipynb` tamamlanmış (`manifest.csv` var)
- `Faz2_Split_Onisleme_1fold.ipynb` tamamlanmış (`fold0_train.csv`, `fold0_val.csv` var)

## 1. Ortam ve Import

In [1]:
# Colab'da çalışıyorsa Drive'ı bağla ve bağımlılıkları kur
import sys, os

ON_COLAB = 'google.colab' in sys.modules
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    !pip install -q timm python-dotenv pydicom opencv-python-headless tensorboard

print('Colab:', ON_COLAB)

Colab: False


In [2]:
import os, sys, random
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# Yol ayarları
if ON_COLAB:
    PROJE = Path('/content/drive/MyDrive/Abdomen')
    CODE  = PROJE / 'code'
else:
    PROJE = Path(os.environ.get('TR_ABDOMEN_PROJE', r'D:/makale-pdf/Proje'))
    CODE  = Path(os.environ.get('TR_ABDOMEN_CODE',  r'D:/makale-pdf/Proje/code'))

sys.path.insert(0, str(CODE))

from src.config import (
    SPLIT_DIR, CLS_DATA_DIR, CKPT_DIR, LOG_DIR, SUPER_CLASSES,
    ClsConfig, DEFAULT_CLS
)
from src.device_utils import get_device

print('PROJE :', PROJE)
print('CODE  :', CODE)
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)

PROJE : /Users/ramazanpolat/Desktop/datasets/abdomen
CODE  : /Users/ramazanpolat/Desktop/datasets/abdomen/code
Python: 3.9.6
PyTorch: 2.8.0


## 2. Seed ve Cihaz

In [3]:
# orijinal notebook'taki set_seed() fonksiyonunun birebir karşılığı
def set_seed(seed: int = 42) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = get_device(verbose=True)
print('Device:', device)

✅ Apple Silicon MPS (Metal) aktif
   Chip   : Apple M5
   PyTorch: 2.8.0
   Python : 3.9.6

   📌 MPS Optimizasyon İpuçları (M5):
   • AMP: bfloat16 kullanın (float16 desteklenmiyor)
   • DataLoader: spawn context + 4-6 worker (fork sorunu önlenir)
   • Batch size: 48+ (unified memory büyükse artırın)
   • PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 → OOM önleme
   • torch.mps.empty_cache() her epoch sonrası

Device: mps


## 3. Split Kontrolü

In [4]:
from src.splits import load_fold, load_holdout
import pandas as pd

fold0_train = load_fold(fold=0, split='train')
fold0_val   = load_fold(fold=0, split='val')
holdout     = load_holdout()

print(f'Fold 0 eğitim vakası : {len(fold0_train)}')
print(f'Fold 0 val vakası    : {len(fold0_val)}')
print(f'Hold-out vakası      : {len(holdout)}')
assert not (set(fold0_train) & set(fold0_val)), 'Train-Val çakışması!'
print('Split doğrulandı ✓')

Fold 0 eğitim vakası : 580
Fold 0 val vakası    : 145
Hold-out vakası      : 129
Split doğrulandı ✓


## 4. Sınıf Dağılımı

Orijinal notebook'taki `Counter` + `WeightedRandomSampler` adımının karşılığı.  
Bizde sınıf dengesizliği `FocalBCE(alpha=class_balanced)` ile ele alınır.

In [5]:
# Sınıf dağılımı — şimdilik devre dışı, aktif etmek için yorumu kaldır
manifest = pd.read_csv(SPLIT_DIR / 'manifest.csv')
manifest['super_labels'] = manifest['super_labels'].fillna('').astype(str)

def count_classes(cases):
    sub = manifest[manifest['case'].isin(cases)]
    cnt = {s: 0 for s in range(len(SUPER_CLASSES))}
    for v in sub['super_labels']:
        for s in v.split(';'):
            if s.strip(): cnt[int(s)] += 1
    return cnt

rows = []
for name, cases in [('train', fold0_train), ('val', fold0_val)]:
    cnt = count_classes(cases)
    rows.append([name, len(cases)] + [cnt[i] for i in range(len(SUPER_CLASSES))])

dist = pd.DataFrame(rows, columns=['split', 'n_case'] + SUPER_CLASSES)
print(dist.to_string(index=False))
print("Sınıf dağılımı devre dışı — aktif etmek için bu hücredeki yorumları kaldırın.")

split  n_case  acute_cholecystitis  kidney_ureter_stone  acute_pancreatitis  aortic_aneurysm_dissection  acute_appendicitis  acute_diverticulitis
train     580                 4622                 1642                6305                        7675                1260                   232
  val     145                 1148                  394                1361                        2252                 404                   109
Sınıf dağılımı devre dışı — aktif etmek için bu hücredeki yorumları kaldırın.


## 5. NPZ Slice Export

Orijinal notebook: `datasets.ImageFolder` ile hazır PNG okunuyor.  
Bizde: DICOM → HU pencereleme → 3-kanallı float16 NPZ.

Bu adım **bir kez** çalıştırılır; dosyalar zaten varsa atlanır.

In [6]:
from src.preprocessing import export_slices

existing = list(CLS_DATA_DIR.glob('*.npz'))
print(f'Mevcut NPZ: {len(existing)}')

if len(existing) == 0:
    print('NPZ bulunamadı — export başlatılıyor...')
    print('(İlk çalıştırmada ~10-30 dk sürebilir)')
    export_slices(
        manifest_csv=SPLIT_DIR / 'manifest.csv',
        out_dir=CLS_DATA_DIR,
        include_negative_sampling=0,   # her vaka için 2 negatif kesit ekle
    )
    existing = list(CLS_DATA_DIR.glob('*.npz'))
    print(f'Export tamamlandı. Toplam NPZ: {len(existing)}')
else:
    print('NPZ dosyaları zaten mevcut, export atlanıyor ✓')

# Örnek NPZ kontrolü
sample = np.load(str(existing[0]))
print(f'\nÖrnek NPZ → image: {sample["image"].shape}, labels: {sample["labels"]}')

Mevcut NPZ: 39268
NPZ dosyaları zaten mevcut, export atlanıyor ✓

Örnek NPZ → image: (512, 512, 3), labels: [0 0 1 0 0 0]


## 6. Swin Transformer Konfigürasyonu

Orijinal: `swin_tiny_patch4_window7_224` — 28M parametre  
Bizde: `swin_base_patch4_window12_384.ms_in22k_ft_in1k` — 88M parametre, ImageNet-22k pretrained  
Giriş: 384×384 — window12 ile tam uyumlu (384/12=32, pad gerekmez)

In [7]:
import dataclasses

# Mevcut ConvNeXt config'inden türet; window12/384 için backbone ve input_size değiştir
swin_cfg = dataclasses.replace(
    DEFAULT_CLS,
    backbone   = 'swin_base_patch4_window12_384.ms_in22k_ft_in1k',
    input_size = 384,          # window12 ile tam uyumlu (384/12=32)
    batch_size = 16,           # 384² → 224²'nin ~3×; M5 belleğine göre artırılabilir
    epochs     = 50,
    lr         = 2e-4,
)

print('=== Swin Transformer Config ===')
for k, v in dataclasses.asdict(swin_cfg).items():
    print(f'  {k:<20}: {v}')

=== Swin Transformer Config ===
  backbone            : swin_base_patch4_window12_384.ms_in22k_ft_in1k
  num_classes         : 6
  input_size          : 384
  batch_size          : 16
  epochs              : 50
  lr                  : 0.0002
  weight_decay        : 0.0001
  warmup_epochs       : 3
  use_focal           : True
  focal_gamma         : 2.0
  mixup_alpha         : 0.2
  accum_steps         : 1
  precision           : bf16-mixed


## 7. Model Oluşturma ve Parametre Sayısı

In [8]:
from src.train_cls import build_model

model = build_model(swin_cfg)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
n_train  = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f'Model    : {swin_cfg.backbone}')
print(f'Toplam   : {n_params:.1f}M parametre')
print(f'Eğitilebilir: {n_train:.1f}M parametre')
print(f'Çıkış    : {swin_cfg.num_classes} sınıf (multi-label sigmoid)')
del model  # eğitim fonksiyonu tekrar oluşturacak

Model    : swin_base_patch4_window12_384.ms_in22k_ft_in1k
Toplam   : 86.9M parametre
Eğitilebilir: 86.9M parametre
Çıkış    : 6 sınıf (multi-label sigmoid)


## 8. Eğitim (Fold 0)

Orijinal notebook'taki `train_model()` fonksiyonunun karşılığı.  
Bizim `train_one_fold()` fonksiyonu şunları içeriyor:
- AMP (CUDA: bfloat16 GradScaler, MPS: autocast)
- Warmup → CosineAnnealing scheduler
- Gradient accumulation
- FocalBCE class-balanced loss
- TensorBoard logging
- En iyi model checkpoint (mAP bazlı)

In [9]:
from src.train_cls import train_one_fold

print('TensorBoard başlatmak için yeni terminal:')
print(f'  tensorboard --logdir "{LOG_DIR / "tb"}"')
print()

best = train_one_fold(fold=0, cfg=swin_cfg)

print('\n=== En iyi sonuç ===')
print(f'  Epoch  : {best["epoch"]}')
print(f'  mAP    : {best["mAP"]:.4f}')
print(f'  macroF1: {best["macroF1"]:.4f}')

TensorBoard başlatmak için yeni terminal:
  tensorboard --logdir "/Users/ramazanpolat/Desktop/datasets/abdomen/outputs/logs/tb"

✅ Apple Silicon MPS (Metal) aktif
   Chip   : Apple M5
   PyTorch: 2.8.0
   Python : 3.9.6

   📌 MPS Optimizasyon İpuçları (M5):
   • AMP: bfloat16 kullanın (float16 desteklenmiyor)
   • DataLoader: spawn context + 4-6 worker (fork sorunu önlenir)
   • Batch size: 48+ (unified memory büyükse artırın)
   • PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 → OOM önleme
   • torch.mps.empty_cache() her epoch sonrası

[fold 0] pos counts: [4622, 1642, 6305, 7675, 1260, 232]
[fold 0] alpha     : ['0.199', '0.377', '0.164', '0.146', '0.437', '0.800']


📊 TensorBoard log: /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/logs/tb/cls_fold0
   tensorboard --logdir /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/logs/tb

  EĞİTİM BAŞLIYOR  │  Fold 0  │  50 epoch
  Backbone  : swin_base_patch4_window12_384.ms_in22k_ft_in1k
  Device    : mps  (workers=6, ctx=spawn, pin=False, prefetch=4)
  Batch     : 16  (accum=1 → eff=16)
  Warmup    : 3 epoch → CosineAnnealing 47 epoch
  LR        : 0.0002  →  2.00e-07
  Train/Val : 24067/6220 slice



[F0] Ep 01/50: 100%|██████████| 1504/1504 [42:52<00:00,  1.71s/bat, loss=0.0389, lr=2.00e-07, ETA=35s01d06s]


─────────────────────────────────────────────────────
  Fold 0  Epoch 00  │  mAP=0.2539  macroF1=0.0000

Sınıf                                    AP       F1
─────────────────────────────────────────────────────
  acute_cholecystitis                0.2642   0.0000
  kidney_ureter_stone                0.0764   0.0000
  acute_pancreatitis                 0.2793   0.0000
  aortic_aneurysm_dissection         0.4367   0.0000
  acute_appendicitis                 0.4496   0.0000
  acute_diverticulitis               0.0172   0.0000
─────────────────────────────────────────────────────
  ⏱  Epoch süresi: 2820s  │  Toplam: 46d59s  │  Kalan: 38s22d49s
  ✅ Yeni en iyi kaydedildi → mAP=0.2539


[F0] Ep 02/50: 100%|██████████| 1504/1504 [42:35<00:00,  1.70s/bat, loss=0.0103, lr=6.68e-05, ETA=35s50d27s]


...

─────────────────────────────────────────────────────
  Fold 0  Epoch 20  │  mAP=0.7436  macroF1=0.6120

Sınıf                                    AP       F1
─────────────────────────────────────────────────────
  acute_cholecystitis                0.8462   0.7692
  kidney_ureter_stone                0.8180   0.7053
  acute_pancreatitis                 0.9169   0.8374
  aortic_aneurysm_dissection         0.9538   0.8502
  acute_appendicitis                 0.7558   0.5096
  acute_diverticulitis               0.1711   0.0000
─────────────────────────────────────────────────────
  ⏱  Epoch süresi: 2839s  │  Toplam: 16s19d02s  │  Kalan: 22s32d00s


[F0] Ep 22/50: 100%|██████████| 1504/1504 [43:08<00:00,  1.72s/bat, loss=0.0009, lr=1.36e-04, ETA=21s40d57s]


─────────────────────────────────────────────────────
  Fold 0  Epoch 21  │  mAP=0.7737  macroF1=0.6376

Sınıf                                    AP       F1
─────────────────────────────────────────────────────
  acute_cholecystitis                0.8310   0.7347
  kidney_ureter_stone                0.7868   0.7273
  acute_pancreatitis                 0.8997   0.7139
  aortic_aneurysm_dissection         0.9594   0.8575
  acute_appendicitis                 0.6602   0.4248
  acute_diverticulitis               0.5049   0.3673
─────────────────────────────────────────────────────
  ⏱  Epoch süresi: 2826s  │  Toplam: 17s06d08s  │  Kalan: 21s45d59s
  ✅ Yeni en iyi kaydedildi → mAP=0.7737


[F0] Ep 23/50:  11%|█         | 167/1504 [04:50<38:46,  1.74s/bat, loss=0.0008, lr=1.30e-04, ETA=21s40d23s]


KeyboardInterrupt: 

In [10]:
# ── TANI 2: NPZ dosyaları manifest'te var mı? Etiket eşleşiyor mu? ─────────
import numpy as np
from pathlib import Path
from src.splits import load_fold
from src.config import CLS_DATA_DIR, SUPER_CLASSES, SPLIT_DIR
import pandas as pd

manifest = pd.read_csv(SPLIT_DIR / 'manifest.csv')
manifest['case'] = manifest['case'].astype(str)
manifest_lookup = {
    (str(r['case']), int(r['image_id'])): r['super_labels']
    for _, r in manifest.iterrows()
}

for split_name, fold_split in [('val', 'val'), ('train', 'train')]:
    cases = load_fold(fold=0, split=fold_split)
    case_set = set(str(c) for c in cases)
    files = sorted(
        p for p in CLS_DATA_DIR.glob("*.npz")
        if p.stem.rsplit("_", 1)[0] in case_set
    )
    in_manifest_pos, in_manifest_neg, not_in_manifest = 0, 0, 0
    for f in files:
        with np.load(f, allow_pickle=False) as npz:
            key = (str(npz['case']), int(npz['image_id']))
        sl = manifest_lookup.get(key, None)
        if sl is None:
            not_in_manifest += 1
        elif str(sl).strip() and str(sl) != 'nan':
            in_manifest_pos += 1
        else:
            in_manifest_neg += 1
    print(f"\n[{split_name.upper()}] {len(files)} NPZ dosyası ({len(cases)} case)")
    print(f"  Manifest'te POZİTİF etiket : {in_manifest_pos}")
    print(f"  Manifest'te SIFIR etiket   : {in_manifest_neg}")
    print(f"  Manifest'te YOK (negatif)  : {not_in_manifest}")


[VAL] 6220 NPZ dosyası (145 case)
  Manifest'te POZİTİF etiket : 5470
  Manifest'te SIFIR etiket   : 750
  Manifest'te YOK (negatif)  : 0

[TRAIN] 24067 NPZ dosyası (580 case)
  Manifest'te POZİTİF etiket : 20964
  Manifest'te SIFIR etiket   : 3103
  Manifest'te YOK (negatif)  : 0


## 9. TensorBoard

In [11]:
%load_ext tensorboard
%tensorboard --logdir {str(LOG_DIR / 'tb')}

## 10. Test Seti Değerlendirme

Orijinal notebook'taki `evaluate_model()` fonksiyonunun multi-label karşılığı.

In [12]:
from src.train_cls import build_model, evaluate
from src.datasets import SliceMultiLabelDataset
from src.splits import load_holdout
from torch.utils.data import DataLoader

# En iyi checkpoint'i yükle
ckpt_path = CKPT_DIR / 'cls_fold0' / 'best.pth'
assert ckpt_path.exists(), f'Checkpoint bulunamadı: {ckpt_path}'

model = build_model(swin_cfg).to(device)
ckpt  = torch.load(str(ckpt_path), map_location=device)
model.load_state_dict(ckpt['model'])
print(f'Checkpoint yüklendi → epoch={ckpt["epoch"]}, mAP={ckpt["metrics"]["mAP"]:.4f}')

# Hold-out değerlendirme
holdout_cases = load_holdout()
holdout_ds    = SliceMultiLabelDataset(holdout_cases, input_size=swin_cfg.input_size)
holdout_loader = DataLoader(holdout_ds, batch_size=swin_cfg.batch_size, shuffle=False)

metrics = evaluate(model, holdout_loader, device)

print('\n=== Hold-out Sonuçları ===')
print(f'  mAP    : {metrics["mAP"]:.4f}')
print(f'  macroF1: {metrics["macroF1"]:.4f}')
print()
print(f'  {"Sınıf":<35} {"AP":>7}  {"F1":>7}')
print('  ' + '-'*55)
for cls in SUPER_CLASSES:
    ap = metrics.get(f'AP/{cls}', float('nan'))
    f1 = metrics.get(f'F1/{cls}', float('nan'))
    print(f'  {cls:<35} {ap:>7.4f}  {f1:>7.4f}')

Checkpoint yüklendi → epoch=21, mAP=0.7737



=== Hold-out Sonuçları ===
  mAP    : 0.7749
  macroF1: 0.6794

  Sınıf                                    AP       F1
  -------------------------------------------------------
  acute_cholecystitis                  0.8927   0.8134
  kidney_ureter_stone                  0.8648   0.8158
  acute_pancreatitis                   0.9148   0.7043
  aortic_aneurysm_dissection           0.9895   0.9570
  acute_appendicitis                   0.8985   0.7157
  acute_diverticulitis                 0.0889   0.0702


## Özet

| Çıktı | Yol |
|---|---|
| NPZ slice'lar | `outputs/cls_data/*.npz` |
| En iyi checkpoint | `outputs/checkpoints/cls_fold0/best.pth` |
| Eğitim logu | `outputs/logs/cls_fold0.csv` |
| TensorBoard | `outputs/logs/tb/cls_fold0/` |

**Sonraki:** Diğer foldlar için `fold` parametresini 1–4 yaparak tekrar çalıştırın.